# Chapter 1: Messages and Prompts

Builds on `basics.ipynb`: same NVIDIA endpoint through `ChatOpenAI`, now passing structured messages and reusable prompt templates. Needs `OPENAI_API_KEY` (your NVIDIA key) in `.env`.

## Setup: the model
`ChatOpenAI` pointed at NVIDIA's OpenAI-compatible endpoint. The key is read from the `OPENAI_API_KEY` env var.

In [10]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    # api_key=os.getenv("OPENAI_API_KEY"),
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
)

## Messages
Instead of one string, pass a list of message objects to `llm.invoke`:
- `SystemMessage`: sets the model's role/behavior.
- `HumanMessage`: the user's input.
- `AIMessage`: a previous model reply (not used here, but needed to replay conversation history).

`.content` on the result is the reply text.

In [2]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

my_messages = [
    SystemMessage(content="You are a coding assistant who provides hints for a coding solution?"),
    HumanMessage(content="What is the approach taken in a 2 sum coding question")
]

llm.invoke(my_messages).content

'The **Two Sum** problem typically has a few standard approaches, ordered from simplest to most optimal. Here\'s the typical breakdown:\n\n### 1. Brute Force 🟢\n- **Idea:** Check every possible pair of numbers using nested loops.\n- **Code idea:** \n  ```python\n  for i in range(len(nums)):\n      for j in range(i+1, len(nums)):\n          if nums[i] + nums[j] == target:\n              return [i, j]\n  ```\n- **Complexity:** `O(n²)` time, `O(1)` space\n- **When to use:** Only for learning/very small inputs. Not interview-optimal.\n\n### 2. Hash Map (Single Pass) 🟢🟢 *Standard interview answer*\n- **Idea:** Store each number\'s index in a dictionary as you iterate. For each `num`, check if `target - num` has already been seen.\n- **Code idea:** \n  ```python\n  seen = {}\n  for i, n in enumerate(nums):\n      complement = target - n\n      if complement in seen:\n          return [seen[complement], i]\n      seen[n] = i\n  ```\n- **Complexity:** `O(n)` time, `O(n)` space\n- **Why it\'s p

## Prompts
Templates let you reuse one prompt with different inputs. Flow: **template -> `.invoke({vars})` -> ready prompt -> `llm.invoke(ready_prompt)`**.

### `PromptTemplate` (single string)
`{topic}` is a placeholder. The next three cells show the steps one at a time: build the template, fill it in, send it to the model. `input()` asks for the topic when the cell runs.

In [4]:
from langchain_core.prompts import PromptTemplate

user_input = input("enter the topic: ")
dynamic_prompt = PromptTemplate.from_template("write a fun fact about {topic}")

dynamic_prompt

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='write a fun fact about {topic}')

Filling the template with `.invoke({"topic": ...})` gives a prompt value with the text substituted.

In [11]:
from langchain_core.prompts import PromptTemplate

user_input = input("enter the topic: ")
dynamic_prompt = PromptTemplate.from_template("write a fun fact about {topic}")

dynamic_prompt.invoke({"topic":user_input})

StringPromptValue(text='write a fun fact about AI')

Passing the filled prompt to the model returns the answer.

In [6]:
from langchain_core.prompts import PromptTemplate

user_input = input("enter the topic: ")
dynamic_prompt = PromptTemplate.from_template("write a fun fact about {topic}")

ready_prompt = dynamic_prompt.invoke({"topic":user_input})

llm.invoke(ready_prompt).content

'Fun fact: Pac-Man was originally going to be called **"Puck-Man"** in Japan. The name comes from the Japanese onomatopoeia *paku-paku*, describing the sound of something being chomped. When it was brought over to the U.S., the name was changed to "Pac-Man" because arcade owners worried that vandals could easily scratch the \'P\' off the cabinet, turning it into a profanity. 🎮👻'

### `ChatPromptTemplate` (role-tagged messages)
Same idea, but each entry is a `(role, text)` pair, so one template produces a system + user message list. The first cell only builds and prints the prompt; the second sends it to the model.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a polite assistant"),
    ("user", "Write a fun fact about {topic}")
])

ready_prompt = prompt_template.invoke({"topic":"AI"})

ready_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a polite assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Write a fun fact about AI', additional_kwargs={}, response_metadata={})])

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a polite assistant"),
    ("user", "Write a fun fact about {topic}")
])

ready_prompt = prompt_template.invoke({"topic":"AI"})

llm.invoke(ready_prompt).content

'Here\'s a fun one: The largest AI models today have over **1.8 trillion parameters** — that\'s more than the estimated number of stars in the entire Milky Way galaxy (which has roughly 100–400 billion)!\n\nFeel free to geek out over that next time you\'re marveling at how much "brainpower" fits inside a computer. 🌌🤖'